## Download data from the ERA5 API

This notebook uses `ecmwf-datastores-client` through `from ecmwf.datastores import Client`.

### Step 1: Set up your CDS API credentials

Register or sign in at the [Copernicus Climate Data Store](https://cds.climate.copernicus.eu/), then obtain your API key from the [API key page](https://cds.climate.copernicus.eu/how-to-api).

Create a file named `.ecmwfdatastoresrc` in your home directory (on Windows, usually `C:\\Users\\<username>\\.ecmwfdatastoresrc`) with the following content:

```text
url: https://cds.climate.copernicus.eu/api
key: <your-personal-api-key>
```

Keep this file private. Do not add your API key to the notebook or share it.

### Step 2: Install the required packages

Install the client used by `ERA5_loader.py` and its pandas dependency:

```python
%pip install ecmwf-datastores-client pandas
```

Restart the notebook kernel after installation if the package was not already available.

### ERA5 grid resolution

The current CDS Data Stores request schemas used here do not expose a custom `grid` parameter.
Requests therefore use the native resolution of the selected ERA5 dataset. Leave
`grid_resolution=None` (the default); a custom value is rejected explicitly by the loader.


### CDS API Request Queue Limit

The Copernicus Climate Data Store (CDS) may temporarily reject data retrieval requests when the maximum number of queued requests for a specific dataset has been reached.

This is a server-side limitation and does not necessarily indicate an error in the request parameters or the downloader itself. Even valid requests may be rejected during periods of high server demand.

**Possible error message:**
```text
Maximum number queued requests for this dataset has been reached.
Please configure your scripts to request data sequentially.
```

**What should you do?**

- Make sure that no multiple instances of the downloader are running simultaneously.
- Avoid submitting multiple requests to the same dataset at the same time.
- If a request is rejected due to the queue limit, wait for some time before trying again.
- Do not repeatedly resubmit rejected requests, as this may increase the load on the CDS service.

The downloader uses sequential requests to reduce the likelihood of reaching this limit. However, queue availability also depends on the overall CDS server workload and requests submitted by other users.

**Note:** This limitation is imposed by the CDS infrastructure and is outside the control of this downloader.

### Recommended Time for Data Downloads

CDS request processing times vary depending on the overall system workload. Although there is no officially guaranteed time when the request queue is least busy, downloads may be processed faster during off-peak hours.

**Recommended practices:**

- Consider submitting large data requests during nighttime or early morning hours (UTC), when user activity may be lower.
- Avoid peak daytime periods when possible.
- Check the current CDS queue status before starting large download operations: [CDS Live](https://cds.climate.copernicus.eu/live).
- Monitor the queue status to identify periods of lower activity.
- For large download operations, consider running the script overnight rather than during working hours.

**Important:** Nighttime downloads are not guaranteed to be faster. CDS workload varies continuously and depends on dataset availability, request size, server capacity, and activity from other users. The recommended time window is a practical suggestion, not an official CDS scheduling policy.

### CDS Processing Time and Request Expiration

Even small requests (a few megabytes) may remain in the CDS queue for several hours before processing completes. Additionally, completed results are stored on CDS servers only temporarily and can expire if not downloaded in time.

**Why small requests can take hours:**

- **Shared CDS queue:** The total duration shown in the CDS portal (e.g., several hours) includes the waiting time in the shared user queue (`Accepted` / `Queued`) before a worker slot is assigned, not just the time required to extract the data. When other users submit large multi-terabyte requests ahead of yours, smaller requests must wait in line.
- **MARS tape archive retrieval:** While frequently accessed ERA5 subsets reside on fast spinning disks, requests spanning many years or less common parameter combinations may require retrieving data from the ECMWF MARS tape archive, which significantly increases turnaround time.

**Request expiration (`Expired` status):**

- Completed files are retained in temporary CDS storage for a limited period—typically **24 to 72 hours** after completion (depending on current cache load, with 24 hours as the baseline retention window).
- Once a request expires, its results are deleted from the CDS server and can no longer be downloaded.
- **How this downloader handles expiration:** If a previously `successful` request has expired (`results_ready=False`), `check_era5_status()` and `download_era5_results()` automatically mark it as `expired` in `ERA5_requests.db`. Re-running `submit_era5_data()` with the same parameters will detect the expired status and automatically submit a fresh request.


In [ ]:
# Helper
year = ["1950", "1951",
        "1952", "1953", "1954",
        "1955", "1956", "1957",
        "1958", "1959", "1960",
        "1961", "1962", "1963",
        "1964", "1965", "1966",
        "1967", "1968", "1969",
        "1970", "1971", "1972",
        "1973", "1974", "1975",
        "1976", "1977", "1978",
        "1979", "1980", "1981",
        "1982", "1983", "1984",
        "1985", "1986", "1987",
        "1988", "1989", "1990",
        "1991", "1992", "1993",
        "1994", "1995", "1996",
        "1997", "1998", "1999",
        "2000", "2001", "2002",
        "2003", "2004", "2005",
        "2006", "2007", "2008",
        "2009", "2010", "2011",
        "2012", "2013", "2014",
        "2015", "2016", "2017",
        "2018", "2019", "2020",
        "2021", "2022", "2023",
        "2024", "2025", "2026"]

month = ["01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"]
days =  ["01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ]
times = ["00:00", "06:00", "12:00", "18:00"]  # Four time steps per day
area = [90, -180, -90, 180]  # North data
pressure_level_all = [
    1000, 975, 950, 925, 900, 875, 850, 825, 800, 775,
    750, 700, 650, 600, 550, 500, 450, 400, 350, 300,
    250, 225, 200, 175, 150, 125, 100, 70, 50, 30,
    20, 10, 7, 5, 3, 2, 1
] # All available pressure levels in hPa

In [1]:
from ERA5_loader import *

In [2]:
# Find variables related to temperature
temperature = search_era5_variables("temperature")

Matching ERA5 variables for 'temperature' (level_type=None, parameter_type=None):
------------------------------------------------------------------------------------------------------------------------------------------
                                    name      units                             cds_variable                 level_type     parameter_type
                             Temperature          K                              temperature            pressure_levels      instantaneous
              Lake mix-layer temperature          K               lake_mix_layer_temperature              single_levels      instantaneous
                 Lake bottom temperature          K                  lake_bottom_temperature              single_levels      instantaneous
            Lake total layer temperature          K             lake_total_layer_temperature              single_levels      instantaneous
                    Lake ice temperature          K                     lake_ice_tem

In [3]:
# Search for temperature on pressure levels
temperature_pl = search_era5_variables("temperature", level_type="pressure_levels")

Matching ERA5 variables for 'temperature' (level_type=pressure_levels, parameter_type=None):
-------------------------------------------------------------
       name units cds_variable      level_type parameter_type
Temperature     K  temperature pressure_levels  instantaneous
-------------------------------------------------------------
Note: `cds_variable` is used in CDS API download requests.



In [4]:
temperature_pl = search_era5_variables("rain", level_type="single_levels")

Matching ERA5 variables for 'rain' (level_type=single_levels, parameter_type=None):
-----------------------------------------------------------------------------------------------
                   name          units            cds_variable    level_type     parameter_type
   Convective rain rate kg m**-2 s**-1    convective_rain_rate single_levels      instantaneous
  Large scale rain rate kg m**-2 s**-1   large_scale_rain_rate single_levels      instantaneous
Total column rain water       kg m**-2 total_column_rain_water single_levels vertical_integrals
-----------------------------------------------------------------------------------------------
Note: `cds_variable` is used in CDS API download requests.



In [2]:
out = r"ERA5_test"

In [4]:
# 1. Hourly 2m temperature over Europe (1 week, 4 synoptic hours per day)
# Size: ~28 fields (~1.3 MB in GRIB)
hourly_requests = submit_era5_data(
    output_dir=out,
    variable="2m_temperature",
    start_date="01-07-2024",
    end_date="07-07-2024",
    mode="hourly",
    times=["00:00", "06:00", "12:00", "18:00"],
    area="europe",
    logging_on=True,
)


2026-11-25 16:19:02,612 - download_era5_data - INFO - Sub-daily data for the HRES (stream=oper/wave) the analyses (type=an) are available hourly. The short forecasts, run twice daily from 06 and 18 UTC, provide hourly output forecast steps from 0 to 18 hours (only steps 1 to 12 hours are available on the CDS disks). For the EDA, the sub-daily non-wave data (stream=enda) are available every 3 hours but the sub-daily wave data (stream=ewda) are available hourly in MARS and 3 hourly on the CDS disks.
2026-11-25 16:19:02,613 - download_era5_data - INFO - Area input formats:
  1. Predefined region name, e.g. 'NH' (or 'northern_hemisphere'), 'Europe' (or 'europe'), 'north_atlantic' (or North Atlantic).
  2. Custom coordinates: [North, West, South, East].
  * Longitude range: -180 to 180 degrees.
  * Latitude range: -90 to 90 degrees.
2026-11-25 16:19:02,613 - download_era5_data - INFO - Region 'europe' selected. Area: [72, -25, 34, 45]
2026-11-25 16:19:03,021 - download_era5_data - INFO - Da

In [5]:
# 2. Daily maximum 2m temperature over Ukraine/Eastern Europe (1 month)
# Size: 31 output fields (~1.5 MB in ZIP/NetCDF)
daily_requests = submit_era5_data(
    output_dir=out,
    variable="2m_temperature",
    start_date="01-07-2024",
    end_date="31-07-2024",
    mode="daily",
    daily_statistic="maximum",
    frequency="1_hourly",
    time_zone="utc+02:00",
    area=[53, 22, 44, 41],  # Custom bounding box [North, West, South, East]
    logging_on=True,
)


2026-11-25 16:19:08,138 - download_era5_data - INFO - Selected sampling frequency: 1_hourly
2026-11-25 16:19:08,138 - download_era5_data - INFO - Selected time zone: UTC+02:00
2026-11-25 16:19:08,139 - download_era5_data - INFO - Users can select to calculate daily statistics from 1 hour, 3 hours and 6 hours data. 
2026-11-25 16:19:08,139 - download_era5_data - INFO - Area input formats:
  1. Predefined region name, e.g. 'NH' (or 'northern_hemisphere'), 'Europe' (or 'europe'), 'north_atlantic' (or North Atlantic).
  2. Custom coordinates: [North, West, South, East].
  * Longitude range: -180 to 180 degrees.
  * Latitude range: -90 to 90 degrees.
2026-11-25 16:19:08,538 - download_era5_data - INFO - Dataset authentication: {'id': 000000, 'sub': '<your-sub-id>', 'role': None, 'email': 'user@example.com'}
2026-11-25 16:19:08,633 - download_era5_data - INFO - Dataset: ERA5 post-processed daily statistics on single levels from 1940 to present
2026-11-25 16:19:08,633 - download_era5_data - I

In [6]:
# 3. Monthly averaged geopotential on key tropospheric pressure levels (5 years, global)
# Size: 60 months × 5 levels = 300 fields (~315 MB in GRIB)
monthly_requests = submit_era5_data(
    output_dir=out,
    variable="geopotential",
    start_date="01-01-2020",
    end_date="31-12-2024",
    mode="monthly",
    pressure_levels=[1000, 850, 700, 500, 250],
    logging_on=True,
)


2026-11-25 16:19:15,652 - download_era5_data - INFO - Selected pressure levels (hPa): ['1000', '850', '700', '500', '250']
2026-11-25 16:19:15,652 - download_era5_data - INFO - Monthly averaged reanalysis and Monthly averaged ensemble members data are produced by averaging all daily data. Monthly averages by hour of day constitute the average over all data within the calendar month for every hour (UTC) of the day in case of reanalysis or every third hour (UTC) of the day for ensemble members.
2026-11-25 16:19:16,078 - download_era5_data - INFO - Dataset authentication: {'id': 000000, 'sub': '<your-sub-id>', 'role': None, 'email': 'user@example.com'}
2026-11-25 16:19:16,263 - download_era5_data - INFO - Dataset: ERA5 monthly averaged data on pressure levels from 1940 to present
2026-11-25 16:19:16,263 - download_era5_data - INFO - Dataset ID: reanalysis-era5-pressure-levels-monthly-means
2026-11-25 16:19:16,264 - download_era5_data - INFO - Available period: 1940-01-01 00:00:00+00:00 – 

In [7]:
# 4. Automatic adjustment of unavailable pressure levels (810 hPa -> 800 hPa)
adjusted_requests = submit_era5_data(
    output_dir=out,
    variable="temperature",
    start_date="01-01-2024",
    end_date="31-03-2024",
    mode="monthly",
    pressure_levels=[850, 810, 500],
    adjust_pressure_levels=True,
    area="north_atlantic",
    logging_on=True,
)


2026-11-25 16:19:23,486 - download_era5_data - WARNING - Unavailable pressure levels will be replaced: 810 → 800
2026-11-25 16:19:23,486 - download_era5_data - INFO - Selected pressure levels (hPa): ['850', '800', '500']
2026-11-25 16:19:23,487 - download_era5_data - INFO - Area input formats:
  1. Predefined region name, e.g. 'NH' (or 'northern_hemisphere'), 'Europe' (or 'europe'), 'north_atlantic' (or North Atlantic).
  2. Custom coordinates: [North, West, South, East].
  * Longitude range: -180 to 180 degrees.
  * Latitude range: -90 to 90 degrees.
2026-11-25 16:19:23,487 - download_era5_data - INFO - Region 'north_atlantic' selected. Area: [70, -80, 0, 10]
2026-11-25 16:19:23,488 - download_era5_data - INFO - Monthly averaged reanalysis and Monthly averaged ensemble members data are produced by averaging all daily data. Monthly averages by hour of day constitute the average over all data within the calendar month for every hour (UTC) of the day in case of reanalysis or every third 

In [10]:
request_db = ERA5RequestDatabase()
client = Client()

pending = [
    request for request in request_db.list_requests()
    if request.get("status") in {"accepted", "successful"}
]

for request_info in pending:
    updated_request = check_era5_status(
        client,
        request_info["request_id"],
        request_db,
    )

    if updated_request is None:
        continue

    print(
        updated_request["request_id"],
        updated_request["output_file"],
        updated_request["status"],
    )


6755bc75-8bb2-4464-b900-83acc8addff8 ERA5_test\2m_temperature_single_hourly_20240701-20240707_7e3d70d8.grib successful
53a03780-85ce-4bdb-b386-1275998f9d5a ERA5_test\2m_temperature_single_daily_maximum_20240701-20240731_48cdd69f.zip accepted
37cff0a7-b6b3-4295-8c53-50fc8b432bf6 ERA5_test\geopotential_pressure_monthly_202001-202412_d9971217.grib accepted
7b45c873-2839-4ec4-a651-3ca455b59258 ERA5_test\temperature_pressure_monthly_202401-202403_efea553b.grib accepted


In [11]:
download_era5_results(request_db, logging_on=True)

2026-11-25 16:22:43,705 - download_era5_results - INFO - Request 6755bc75-8bb2-4464-b900-83acc8addff8 downloaded: ERA5_test\2m_temperature_single_hourly_20240701-20240707_7e3d70d8.grib


2026-11-25 16:22:43,932 - download_era5_results - INFO - Request 53a03780-85ce-4bdb-b386-1275998f9d5a is not ready. Current status: accepted
2026-11-25 16:22:44,136 - download_era5_results - INFO - Request 37cff0a7-b6b3-4295-8c53-50fc8b432bf6 is not ready. Current status: accepted
2026-11-25 16:22:44,350 - download_era5_results - INFO - Request 7b45c873-2839-4ec4-a651-3ca455b59258 is not ready. Current status: accepted
2026-11-25 16:22:44,350 - download_era5_results - INFO - Download process completed. Downloaded 1 of 4 requests.


[{'request_id': '6755bc75-8bb2-4464-b900-83acc8addff8',
  'output_file': 'ERA5_test\\2m_temperature_single_hourly_20240701-20240707_7e3d70d8.grib',
  'result': 'ERA5_test\\2m_temperature_single_hourly_20240701-20240707_7e3d70d8.grib',
  'status': 'downloaded'}]